#PREVISÃO DE CARGA DE ENERGIA ELÉTRICA

## 1. Objetivo do projeto

**Problema de negócio:** entender quais fatores explicam a carga de energia elétrica e prever a carga dos próximos 7 dias, identificando os dias de maior demanda esperada.

**Perguntas que o projeto responde:**
1. Como a carga varia por hora, dia da semana e mês?
2. Existem tendência e padrões sazonais?
3. Quais variáveis têm relação com a carga?
4. Um modelo de Machine Learning prevê melhor do que uma previsão sazonal simples?
5. Qual é a carga estimada para os próximos 7 dias e quais dias devem ter maior demanda?

**Definições do projeto:**

| Item | Definição |
|---|---|
| Fonte | ONS — Curva de Carga Horária (dados abertos, licença CC-BY) |
| Recorte | Subsistema Sudeste/Centro-Oeste (SE/CO), jan/2022 até o último dia completo |
| Variável-alvo | Carga média diária, em MWmed |
| Horizonte | 7 dias à frente |
| Validação | Treino até 31/12/2025, teste em 2026 |

> **Carga x consumo:** o ONS mede a *carga*, isto é, a energia que o sistema interligado precisa atender. Ela é diferente do consumo faturado pelas distribuidoras, mas é o indicador usado no planejamento da operação.


In [1]:
#Importando bibliotecas

import truststore
truststore.inject_into_ssl()
from pathlib import Path

import numpy as  np
import pandas as pd
import matplotlib as plt

#Altera os numeros das tabelas para numero decimais com uma casa decimal e separados por virgulas
pd.set_option("display.float_format", "{:,.1f}".format)

In [2]:
#Parâmetros do projeto

URL_BASE = "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/curva-carga-ho/CURVA_CARGA_{ano}.parquet"
ANO_INICIAL = 2022
DATA_EXTRACAO = pd.Timestamp.now().normalize()
ANOS = range(ANO_INICIAL, DATA_EXTRACAO.year + 1)

PASTA_RAW = Path("data/raw")
PASTA_RAW.mkdir(parents=True, exist_ok=True)

print(f"Data de extração: {DATA_EXTRACAO:%d/%m/%Y}")
print(f"Anos: {list(ANOS)}")


Data de extração: 04/11/2026
Anos: [2022, 2023, 2024, 2025, 2026]


In [3]:
partes = []
for ano in ANOS:
    arquivo = PASTA_RAW / f"CURVA_CARGA_{ano}.parquet"
    #Baixa se ainda não existe. O ano corrente é sempre atualizado
    if not arquivo.exists() or ano == DATA_EXTRACAO.year:
        print(f"Baixando {ano}...")
        pd.read_parquet(URL_BASE.format(ano=ano)).to_parquet(arquivo, index=False)
    partes.append(pd.read_parquet(arquivo))

bruto = pd.concat(partes, ignore_index=True)
print(f"Linhas carregadas: {len(bruto):,}")

Baixando 2022...
Baixando 2023...
Baixando 2024...
Baixando 2025...
Baixando 2026...
Linhas carregadas: 169,632


In [4]:
# Entendendo as colunas

bruto.info()
bruto.head()

<class 'pandas.DataFrame'>
RangeIndex: 169632 entries, 0 to 169631
Data columns (total 4 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   id_subsistema            169632 non-null  string        
 1   nom_subsistema           169632 non-null  string        
 2   din_instante             169632 non-null  datetime64[ns]
 3   val_cargaenergiahomwmed  169632 non-null  object        
dtypes: datetime64[ns](1), object(1), string(2)
memory usage: 6.4+ MB


,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed
0,N,NORTE,2022-01-01 00:00:00,5152.56599999
1,NE,NORDESTE,2022-01-01 00:00:00,10508.82200000
2,S,SUL,2022-01-01 00:00:00,10096.15200000
3,SE,SUDESTE,2022-01-01 00:00:00,31822.65600000
4,N,NORTE,2022-01-01 01:00:00,5223.93600000


In [5]:
#Subsistemas presentes na base (código e nome)
bruto.iloc[:, :2].drop_duplicates()

,id_subsistema,nom_subsistema
0,N,NORTE
1,NE,NORDESTE
2,S,SUL
3,SE,SUDESTE
140259,SE,SUDESTE/CENTRO-OESTE


In [6]:
#Ajustando o nome das colunas

COLUNAS = {"id_subsistema": "subsistema",
           "din_instante": "data_hora",
           "val_cargaenergiahomwmed": "carga_mwmed",
}
SUBSISTEMA = "SE"

faltando= set(COLUNAS) - set(bruto.columns)
assert not faltando, f"Colunas não encontradas: {faltando}. Ajuste COLUNAS com os nomes exibidos acima."

horario = (
    bruto[list(COLUNAS)]
    .rename(columns=COLUNAS)
    .query(f"subsistema == @SUBSISTEMA")
    .drop(columns="subsistema")
)

horario["data_hora"]= pd.to_datetime(horario["data_hora"])
horario["carga_mwmed"] = pd.to_numeric(horario["carga_mwmed"], errors="coerce")
horario = horario.sort_values("data_hora").reset_index(drop=True)

print(f"Registros do subsistema {SUBSISTEMA}: {len(horario):,}")
print(f"Periodo: {horario['data_hora'].min()} a {horario['data_hora'].max()}")


Registros do subsistema SE: 42,408
Periodo: 2022-01-01 00:00:00 a 2026-11-02 23:00:00
